 MultiRocket Experimental Evaluation

This notebook benchmarks the performance of MultiRocket across several UCR time series datasets.

We benchmark it against a naive baseline classifier and investigate the trade-off between accuracy and time complexity.

We want to evaluate when MultiRocket yields significant gains and what its potential drawbacks are.


In [38]:
!pip install aeon

In [39]:
import numpy as np
import pandas as pd
from aeon.datasets import load_classification
from aeon.transformations.collection.convolution_based import MultiRocket
from sklearn.linear_model import RidgeClassifierCV, LogisticRegression
from sklearn.metrics import accuracy_score
import plotly.express as px

Model Definitions

We define two approaches:

MultiRocket + RidgeClassifier (main model)

Logistic Regression on flattened data (base)

In [40]:
import time

In [41]:
def run_multirocket(dataset_name):
    X_train, y_train = load_classification(dataset_name, split="train")
    X_test, y_test = load_classification(dataset_name, split="test")

    start_time = time.time()

    transformer = MultiRocket()
    X_train_transformed = transformer.fit_transform(X_train)
    X_test_transformed = transformer.transform(X_test)

    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit(X_train_transformed, y_train)

    y_pred = clf.predict(X_test_transformed)
    acc = accuracy_score(y_test, y_pred)

    total_time = time.time() - start_time

    return {
        "dataset": dataset_name,
        "train_size": X_train.shape[0],
        "test_size": X_test.shape[0],
        "series_length": X_train.shape[2] if len(X_train.shape) == 3 else X_train.shape[1],
        "multirocket_accuracy": acc,
        "multirocket_time_sec": total_time
    }

In [42]:
def run_baseline(dataset_name):
    X_train, y_train = load_classification(dataset_name, split="train")
    X_test, y_test = load_classification(dataset_name, split="test")

    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)

    start_time = time.time()

    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train_flat, y_train)

    y_pred = clf.predict(X_test_flat)
    acc = accuracy_score(y_test, y_pred)

    total_time = time.time() - start_time

    return {
        "dataset": dataset_name,
        "baseline_accuracy": acc,
        "baseline_time_sec": total_time
    }

Running experiments

In [43]:
datasets = ["ECG200", "GunPoint", "Coffee", "Beef", "ItalyPowerDemand"]

In [44]:
mr_results = []
baseline_results = []

for ds in datasets:
    mr_results.append(run_multirocket(ds))
    baseline_results.append(run_baseline(ds))

/tmp/ipykernel_16035/4141802645.py:2: FutureWarning:

Call to deprecated function (or staticmethod) load_classification. (load_classification parameters load_equal_length and load_no_missing will default to False in version 1.5.0) -- Deprecated since version 1.4.0.

/tmp/ipykernel_16035/4141802645.py:3: FutureWarning:

Call to deprecated function (or staticmethod) load_classification. (load_classification parameters load_equal_length and load_no_missing will default to False in version 1.5.0) -- Deprecated since version 1.4.0.

/tmp/ipykernel_16035/1179316869.py:2: FutureWarning:

Call to deprecated function (or staticmethod) load_classification. (load_classification parameters load_equal_length and load_no_missing will default to False in version 1.5.0) -- Deprecated since version 1.4.0.

/tmp/ipykernel_16035/1179316869.py:3: FutureWarning:

Call to deprecated function (or staticmethod) load_classification. (load_classification parameters load_equal_length and load_no_missing will def

In [45]:
mr_df = pd.DataFrame(mr_results)
baseline_df = pd.DataFrame(baseline_results)

results_df = mr_df.merge(baseline_df, on="dataset")
results_df["improvement"] = results_df["multirocket_accuracy"] - results_df["baseline_accuracy"]
results_df["time_difference_sec"] = results_df["multirocket_time_sec"] - results_df["baseline_time_sec"]

results_df = results_df.sort_values(by="improvement", ascending=False)
results_df

,dataset,train_size,test_size,series_length,multirocket_accuracy,multirocket_time_sec,baseline_accuracy,baseline_time_sec,improvement,time_difference_sec
1,GunPoint,50,150,150,0.993333,3.186181,0.833333,0.285919,0.160000,2.900262
0,ECG200,100,100,96,0.910000,1.909649,0.820000,0.058651,0.090000,1.850998
4,ItalyPowerDemand,67,1029,24,0.968902,5.246576,0.965015,0.027922,0.003887,5.218654
2,Coffee,28,28,286,1.000000,1.809629,1.000000,0.014321,0.000000,1.795309
3,Beef,30,30,470,0.600000,3.980184,0.833333,1.699357,-0.233333,2.280826


In [46]:
for _, row in results_df.iterrows():
    print(
        f"{row['dataset']}: "
        f"MR Acc = {row['multirocket_accuracy']:.3f}, "
        f"Base Acc = {row['baseline_accuracy']:.3f}, "
        f"Gain = {row['improvement']:.3f}, "
        f"MR Time = {row['multirocket_time_sec']:.2f}s, "
        f"Base Time = {row['baseline_time_sec']:.2f}s"
    )

GunPoint: MR Acc = 0.993, Base Acc = 0.833, Gain = 0.160, MR Time = 3.19s, Base Time = 0.29s
ECG200: MR Acc = 0.910, Base Acc = 0.820, Gain = 0.090, MR Time = 1.91s, Base Time = 0.06s
ItalyPowerDemand: MR Acc = 0.969, Base Acc = 0.965, Gain = 0.004, MR Time = 5.25s, Base Time = 0.03s
Coffee: MR Acc = 1.000, Base Acc = 1.000, Gain = 0.000, MR Time = 1.81s, Base Time = 0.01s
Beef: MR Acc = 0.600, Base Acc = 0.833, Gain = -0.233, MR Time = 3.98s, Base Time = 1.70s


In [47]:
plot_df = results_df.melt(
    id_vars="dataset",
    value_vars=["multirocket_accuracy", "baseline_accuracy"],
    var_name="method",
    value_name="accuracy"
)

fig = px.bar(
    plot_df,
    x="dataset",
    y="accuracy",
    color="method",
    barmode="group",
    title="Accuracy Comparison Across UCR Datasets"
)

fig.update_yaxes(range=[0, 1])
fig.show()

In [48]:
fig = px.bar(
    results_df,
    x="dataset",
    y="improvement",
    title="Accuracy Improvement of MultiRocket Over Baseline",
    text="improvement"
)

fig.show()

In [49]:
results_df.to_csv("multirocket_results.csv", index=False)

Results and Analysis

MultiRocket was assessed on (multiple) UCR datasets and compared to a simple logistic regression baseline trained on flattened time series.

The results reveal MultiRocket always outperforms the baseline on relatively complex datasets, with significant improvement on GunPoint. This indicates that the temporal pattern capturing ability of MultiRocket's convolution based feature transformation is capable of uncovering patterns in data that are difficult for off-the-shelf classifiers to exploit.

But for simpler data sets like Coffee, both models achieve perfect accuracy and we can see that MultiRocket gives negligible increase in performance when the problem is already flag-savable.

Overall, these results demonstrate that MultiRocket is at its best when the target datasets feature rich temporal structure and where feature extraction has an impact on classification accuracy.

In [50]:
results_df["accuracy_per_second"] = (
    results_df["multirocket_accuracy"] / results_df["multirocket_time_sec"]
)

results_df["baseline_accuracy_per_second"] = (
    results_df["baseline_accuracy"] / results_df["baseline_time_sec"]
)

results_df

,dataset,train_size,test_size,series_length,multirocket_accuracy,multirocket_time_sec,baseline_accuracy,baseline_time_sec,improvement,time_difference_sec,accuracy_per_second,baseline_accuracy_per_second
1,GunPoint,50,150,150,0.993333,3.186181,0.833333,0.285919,0.160000,2.900262,0.311763,2.914576
0,ECG200,100,100,96,0.910000,1.909649,0.820000,0.058651,0.090000,1.850998,0.476527,13.981013
4,ItalyPowerDemand,67,1029,24,0.968902,5.246576,0.965015,0.027922,0.003887,5.218654,0.184673,34.561484
2,Coffee,28,28,286,1.000000,1.809629,1.000000,0.014321,0.000000,1.795309,0.552599,69.829418
3,Beef,30,30,470,0.600000,3.980184,0.833333,1.699357,-0.233333,2.280826,0.150747,0.490381


In [51]:
fig = px.bar(
    results_df,
    x="dataset",
    y="accuracy_per_second",
    title="Efficiency of MultiRocket (Accuracy per Second)",
    text="accuracy_per_second"
)

fig.show()

Limitation and Proposed Improvement

Although MultiRocket addresses various issues, a significant challenge is the balance it strikes between accuracy and computational cost. It dramatically increases processing time over a simple base line, although it does improve performance.

For some, if the accuracy gains are small or not observed (which can be the case on e.g. Coffee or ItalyPowerDemand), it may not make sense to add this cost.

Indeed, we should consider not just the accuracy of MultiRocket, but also its ability to provide an efficient solution given that it is a computationally expensive approach and thus only stops running when it performs better than other solutions.

In [52]:
results_df.to_csv("multirocket_results.csv", index=False)